# 第 6 周练习 ——「价格合适」多提供商基准 (mmaitsimwale)

## 练习目标（理念）

用**同一套零样本（zero-shot）提示**，比较不同 API 提供商上的语言模型：仅凭产品文字描述（`item.summary`），估美元价格能有多准？

## 和本课第 6 周的关系

| 本课天数 | 你会用到的概念 |
|----------|----------------|
| 第 1 天 | 数据：Amazon Reviews 2023 → `Item`（标题、类别、价格、重量、摘要） |
| 第 2 天 | 预处理：用 LLM 把杂乱描述改写成干净的 `item.summary` |
| 第 3 天 | 评估：`pricer.evaluator.Tester` + 随机/均值/回归等基线 |
| 第 4 天 | 前沿模型做零样本定价（GPT / Claude / Gemini / Grok 等） |
| 第 5 天 | 微调：JSONL 上传 OpenAI，训私有 `gpt-4.1-nano` 变体再评估 |

## 本笔记本测什么

两条提供商路线，提示完全相同：

- **OpenRouter**（付费路由）→ OpenAI GPT 变体
- **Groq**（免费 OSS 推理）→ Llama 3.3 / Llama 3.1 / DeepSeek R1 Distill

指标：平均绝对误差 MAE（美元，越低越好）+ R²（越接近 100% 越好），最后打排行榜。

## 怎么跑

1. 从仓库根目录启动 Jupyter，确保能找到 `week6/pricer`
2. `.env` 准备：`HF_TOKEN`、`OR_API_KEY`、`GROQ_API_KEY`（及可选 URL）
3. 自上而下运行；`BENCHMARK_SIZE` 默认 50，可调大以更稳


## 我们在测什么

**任务：**给定 `item.summary`（产品文本描述），预测美元价格。

**指标：**

- **MAE（平均绝对误差）**：在 `BENCHMARK_SIZE` 个测试样本上，预测与真值差的绝对值平均（单位 `$`），越低越好
- **R²**：预测与真实价格的相关程度，接近 100% 表示「方向对、趋势对」

**本练习对比的提供商 / 模型：**

| 提供商 | 模型 ID（勿改，需与 API 一致） | 类型 |
|--------|--------------------------------|------|
| OpenRouter | `openai/gpt-4.1-nano` | 付费（小、快） |
| OpenRouter | `openai/gpt-4o-mini` | 付费（成本与精度较均衡） |
| Groq | `llama-3.3-70b-versatile` | 免费 OSS（大） |
| Groq | `llama-3.1-8b-instant` | 免费 OSS（小、快） |
| Groq | `deepseek-r1-distill-llama-70b` | 免费 OSS（推理向） |

**凭证（Environment Variables）：**

- OpenRouter：`OR_API_KEY` + `OR_CLIENT_URL`
- Groq：`GROQ_API_KEY` + `GROQ_CLIENT_URL`

两家都兼容 OpenAI SDK，因此统一用 `OpenAI(api_key=..., base_url=...)`。


In [ ]:
# ========== 导入 + 仓库根目录：把 week6/pricer 放进可导入路径 ==========

# 标准库：环境变量、正则、sys.path、JSON、路径对象
import os
import re
import sys
import json
from pathlib import Path
# load_dotenv：把 .env 密钥读进进程环境，避免把密钥写进笔记本
from dotenv import load_dotenv
# OpenAI 兼容客户端：后面 OpenRouter / Groq 都用同一套 SDK
from openai import OpenAI
# r2_score：算 R²，用来衡量预测与真价的相关程度
from sklearn.metrics import r2_score
# pandas：本格导入备用（后续排行榜主要用纯 Python 打印）
import pandas as pd


def _repo_root() -> Path:
    """Walk up from cwd until week6/pricer is found."""
    # 先看 cwd，再看所有父目录候选
    for cand in [Path.cwd(), *Path.cwd().parents]:
        # 课程约定：仓库根下应有 week6/pricer
        if (cand / "week6" / "pricer").is_dir():
            return cand
    # 找不到就明确报错：提醒从 llm_engineering 根目录启动 Jupyter
    raise RuntimeError(
        "Cannot find week6/pricer — start Jupyter from the llm_engineering repo root."
    )


# 解析出仓库根 Path
REPO_ROOT = _repo_root()

# 把 week6/ 插到 sys.path 最前，这样任意 CWD 都能 from pricer...
WEEK6_PATH = str(REPO_ROOT / "week6")
if WEEK6_PATH not in sys.path:
    sys.path.insert(0, WEEK6_PATH)

# Item：产品对象；Tester：并行评测 + 误差可视化
from pricer.items import Item
from pricer.evaluator import Tester

# 每个模型评多少条；想更稳可改大到 200（成本/时间也会上去）
BENCHMARK_SIZE = 50   # items per model — raise to 200 for a more reliable run
# Tester 内部线程数：并行打 API，加快基准
WORKERS = 3

# 打印关键路径与基准配置，方便核对是不是从对的目录跑的
print(f"Repo root : {REPO_ROOT}")
print(f"Week6 path: {WEEK6_PATH}")
print(f"Benchmark : {BENCHMARK_SIZE} items per model, {WORKERS} workers")


In [ ]:
# ========== 读 .env 并检查密钥是否存在（不打印完整密钥）==========

# override=True：笔记本里的值覆盖已有环境变量，避免旧值干扰
load_dotenv(REPO_ROOT / ".env", override=True)

# HuggingFace：拉数据集用
hf_token         = os.getenv("HF_TOKEN")
# OpenAI 官方密钥：本练习主路径走 OpenRouter/Groq，这里仍检查是否配置
openai_api_key   = os.getenv("OPENAI_API_KEY")
# OpenRouter：付费路由
or_api_key       = os.getenv("OR_API_KEY")
# 默认 OpenRouter OpenAI-compatible base URL（字符串勿改，除非你有自定义网关）
or_client_url    = os.getenv("OR_CLIENT_URL", "https://openrouter.ai/api/v1")
# Groq：OSS 快速推理
groq_api_key     = os.getenv("GROQ_API_KEY")
# 默认 Groq OpenAI-compatible base URL
groq_client_url  = os.getenv("GROQ_CLIENT_URL", "https://api.groq.com/openai/v1")

# 逐项检查：有则只显示前 8 字符，无则提示该路线不可用
for name, val in [
    ("HF_TOKEN",        hf_token),
    ("OPENAI_API_KEY",  openai_api_key),
    ("OR_API_KEY",      or_api_key),
    ("GROQ_API_KEY",    groq_api_key),
]:
    if val:
        print(f"{name} exists and begins {val[:8]}")
    else:
        print(f"{name} NOT SET — some models will be unavailable")


In [ ]:
# ========== 两个 OpenAI 兼容客户端：一家提供商一个 ==========

# OpenRouter 客户端：api_key + base_url 指向 OpenRouter
or_client = OpenAI(
    api_key=or_api_key,
    base_url=or_client_url,
)

# Groq 客户端：同样 SDK，换 base_url 就能打到 Groq
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url=groq_client_url,
)

# 打印实际 base_url，确认没指错网关
print("OpenRouter client  :", or_client.base_url)
print("Groq client        :", groq_client.base_url)


In [ ]:
# ========== 从 HuggingFace Hub 加载 Day1 精选商品数据 ==========
# lite：约 22k train / 1k val / 1k test，成本低；要更有代表性可改用 ed-donner/items_full

# login：把 HF_TOKEN 交给 huggingface_hub（下载私有/限速更稳）
from huggingface_hub import login

if hf_token:
    # add_to_git_credential=True：顺带写 git 凭证（与课程其它笔记本一致）
    login(hf_token, add_to_git_credential=True)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN — cannot load dataset")

# 数据集 ID：字符串影响拉到哪份数据，保持原文
DATASET = "ed-donner/items_lite"

# Item.from_hub：课程封装，返回 train / val / test 三个列表
train, val, test = Item.from_hub(DATASET)
# 规模与样例：确认加载成功、summary 长什么样
print(f"Loaded  {len(train):,} train | {len(val):,} val | {len(test):,} test items")
print(f"Sample  : {test[0]}")
print(f"Summary : {test[0].summary[:120]}...")


In [ ]:
# ========== 零样本 messages：与 day4 同一套路 —— 只给 user 一条 ==========

def messages_for(item: Item) -> list[dict]:
    """Build the messages list for zero-shot price estimation."""
    return [
        {
            "role": "user",
            # 提示词字符串影响模型行为：保持英文原文，不翻译
            "content": (
                "Estimate the price of this product. "
                "Respond with the price only, no explanation.\n\n"
                + item.summary
            ),
        }
    ]


# 用第一条测试样本做健全性检查：看消息结构 + 真实价格
print(messages_for(test[0]))
print(f"\nActual price: ${test[0].price:.2f}")


In [ ]:
# ========== Pricer 工厂：为任意 OpenAI 兼容客户端生成「可命名」定价函数 ==========
# Tester 会用函数的 __name__ 当展示标题，所以闭包里要改掉 __name__ / __qualname__

def make_pricer(client: OpenAI, model: str, max_tokens: int = 10):
    """
    Return a callable `fn(item) -> str` that calls `client` with `model`.

    The function name is set to the model slug so Tester.make_title() renders it nicely.
    """
    def pricer(item: Item) -> str:
        # Chat Completions：模型 ID、messages、max_tokens 都保持原参数
        response = client.chat.completions.create(
            model=model,
            messages=messages_for(item),
            max_tokens=max_tokens,
        )
        # 只取第一条 choice 的文本内容（通常是价格字符串）
        return response.choices[0].message.content

    # 把 / - . 换成 _，得到合法、可读的函数名
    pricer.__name__ = model.replace("/", "_").replace("-", "_").replace(".", "_")
    pricer.__qualname__ = pricer.__name__
    return pricer


# ── OpenRouter（付费）────────────────────────────────────────────────────────
gpt_4_1_nano_or = make_pricer(or_client,   "openai/gpt-4.1-nano")
gpt_4o_mini_or  = make_pricer(or_client,   "openai/gpt-4o-mini")

# ── Groq（免费 OSS）──────────────────────────────────────────────────────────
llama_33_70b        = make_pricer(groq_client, "llama-3.3-70b-versatile")
llama_31_8b         = make_pricer(groq_client, "llama-3.1-8b-instant")
deepseek_r1_distill = make_pricer(groq_client, "deepseek-r1-distill-llama-70b")

# 注册表：显示名 → (定价函数, 提供商标签)；循环基准时按这个字典遍历
PRICERS: dict[str, tuple] = {
    "GPT-4.1-nano (OpenRouter)":     (gpt_4_1_nano_or,    "OpenRouter"),
    "GPT-4o-mini (OpenRouter)":      (gpt_4o_mini_or,     "OpenRouter"),
    "Llama-3.3-70b (Groq)":          (llama_33_70b,       "Groq"),
    "Llama-3.1-8b (Groq)":           (llama_31_8b,        "Groq"),
    "DeepSeek-R1-Distill (Groq)":    (deepseek_r1_distill,"Groq"),
}

# 列出已注册定价器，确认 5 条路线都就位
print(f"Registered {len(PRICERS)} pricers:")
for display_name, (fn, provider) in PRICERS.items():
    print(f"  [{provider}] {display_name}")


In [ ]:
# ========== 跑基准：每个模型各跑一轮 Tester ==========
# Tester 用 WORKERS 线程并行评 BENCHMARK_SIZE 条，并画误差趋势/散点

# 收集每个显示名对应的指标与原始序列，供下一格排行榜使用
benchmark_results: dict[str, dict] = {}

for display_name, (fn, provider) in PRICERS.items():
    # 分隔线：肉眼区分不同模型的日志块
    print(f"\n{'='*60}")
    print(f"  {display_name}")
    print(f"{'='*60}")
    try:
        # title= 覆盖默认函数名标题；size/workers 控制样本数与并行度
        t = Tester(fn, test, title=display_name, size=BENCHMARK_SIZE, workers=WORKERS)
        # 真正打 API、填 t.errors / t.truths / t.guesses
        t.run()
        # MAE：绝对误差均值
        avg_err = sum(t.errors) / len(t.errors)
        # R² 乘 100，后面排行榜按百分数展示
        r2      = r2_score(t.truths, t.guesses) * 100
        benchmark_results[display_name] = {
            "provider": provider,
            "avg_error": avg_err,
            "r2": r2,
            "errors": t.errors,
            "guesses": t.guesses,
            "truths": t.truths,
        }
    except Exception as exc:
        # 某模型失败（缺钥、限流等）不中断整场基准；记成 inf 方便排到最后
        print(f"  ERROR: {exc}")
        benchmark_results[display_name] = {
            "provider": provider,
            "avg_error": float("inf"),
            "r2": float("-inf"),
            "errors": [],
            "guesses": [],
            "truths": [],
        }

print(f"\nBenchmark complete. {len(benchmark_results)} models evaluated.")


In [ ]:
# ========== 排行榜：按平均绝对误差升序（越低越好）==========

rows = []
for display_name, r in benchmark_results.items():
    if r["avg_error"] == float("inf"):
        # 失败模型：表格里显示 ERROR，R² 用破折号
        row = {
            "Model": display_name,
            "Provider": r["provider"],
            "Avg Error ($)": "ERROR",
            "R² (%)": "—",
        }
    else:
        # 成功：格式化成两位/一位小数的字符串，便于对齐打印
        row = {
            "Model": display_name,
            "Provider": r["provider"],
            "Avg Error ($)": f"{r['avg_error']:.2f}",
            "R² (%)": f"{r['r2']:.1f}",
        }
    rows.append(row)

# 排序键：能转 float 的按数值排；ERROR 等转不成 → +inf 沉底
def sort_key(row):
    try:
        return float(row["Avg Error ($)"])
    except (ValueError, TypeError):
        return float("inf")

rows.sort(key=sort_key)

# 纯文本排行榜表头
print(f"\n{'='*70}")
print(f"{'LEADERBOARD':^70}")
print(f"{'='*70}")
print(f"{'Rank':<5} {'Model':<35} {'Provider':<12} {'Avg Err ($)':>12} {'R²':>8}")
print("-" * 70)

# 逐行打印名次
for rank, row in enumerate(rows, 1):
    err = row["Avg Error ($)"]
    r2  = row["R² (%)"]
    print(f"{rank:<5} {row['Model']:<35} {row['Provider']:<12} {err:>12} {r2:>8}")

print("-" * 70)

# 若第一名不是 ERROR，额外打印冠军摘要
if rows and rows[0]["Avg Error ($)"] != "ERROR":
    winner = rows[0]
    print(f"\nBest model : {winner['Model']}")
    print(f"Provider   : {winner['Provider']}")
    print(f"Avg Error  : ${winner['Avg Error ($)']}")
    print(f"R²         : {winner['R² (%)']}")


## 主要发现（读结果时对照）

- **OpenRouter**：请求会路由到官方 OpenAI 端点；对「只要一个数」的结构化任务，`GPT-4o-mini` 常常在成本与精度之间较均衡。
- **Groq**：给开源权重模型做极快推理；价格估算上，`Llama-3.3-70b` 往往是较强的 OSS 选项。
- **DeepSeek-R1-Distill**：推理模型可能对简单回归「想太多」，输出偏长；`Tester` 会抓它找到的第一个数字，但仍可能解析失败。
- **对照第 5 天微调**：在同一数据上用少量样本微调的 `gpt-4.1-nano` 变体，通常仍能压过零样本，说明任务专用微调即使数据很少也有价值。

### 后续步骤（第 5 天方向）

要把训练样本变成微调数据：转成 JSONL，再上传到 OpenAI（或你选的提供商）。下一格只演示格式，不会真正上传。


In [ ]:
# ========== 第 5 天预览：OpenAI chat 格式的 JSONL（只打印样例，不上传）==========

def make_jsonl(items: list, n: int = 5) -> str:
    """Produce JSONL fine-tuning data in OpenAI chat format."""
    lines = []
    for item in items[:n]:
        # user：与零样本相同的估价提示 + summary；assistant：真价作标签
        messages = [
            {"role": "user", "content": (
                "Estimate the price of this product. "
                "Respond with the price only, no explanation.\n\n"
                + item.summary
            )},
            {"role": "assistant", "content": f"${item.price:.2f}"},
        ]
        # 一行一个 JSON 对象，这就是 JSONL
        lines.append(json.dumps({"messages": messages}))
    return "\n".join(lines)


# 用训练集前 5 条生成样例并打印
sample_jsonl = make_jsonl(train, n=5)
print("=== Sample fine-tuning JSONL (first 5 items) ===\n")
print(sample_jsonl)
print(f"\n... and {len(train):,} more training examples available.")
print("\nTo fine-tune, upload via:")
# 提示下一步 API 调用形态（字符串保持原文，便于复制）
print('  openai.files.create(open("fine_tune_train.jsonl","rb"), purpose="fine-tune")')
print('  openai.fine_tuning.jobs.create(training_file=..., model="gpt-4.1-nano-2025-04-14")')
